# nb1 — Phase 0 (tiếp): Align âm tiết → pseudo-annotation cho bộ test 6k

Notebook thứ hai trong chuỗi nb0→nb3 (DESIGN.md §7): nhận output của nb0, xây **module align âm tiết** (Levenshtein trên chuỗi token canonical), kiểm thử trên case biên đã biết, **kiểm chứng trên gold annotation của toàn bộ VSEC** (3 tầng), align bộ test 6k → pseudo-annotation schema giống VSEC, thống kê câu sạch (trả lời câu hỏi treo DESIGN.md §11.1) và QA 50 mẫu để soát tay.

Tham chiếu: `PROJECT.md` §3.2 (edge case annotation) · `DESIGN.md` §2 (module align + QA alignment), §6 (align là chỗ dễ sai nhất — phải kiểm thử kỹ), §9 (checklist leakage).

**Input**: output nb0 — trên Kaggle: *Add Input* dataset tạo từ nb0 (file nằm trong `/kaggle/input/<tên-dataset>/`); local: `./out`. **Không cần Internet**, không cài package mới (chỉ stdlib — mọi trường VSEC cần thiết đã có trong jsonl của nb0).

**Nguyên tắc leakage (DESIGN.md §9)**: align trên test chỉ phục vụ **đánh giá** (tạo pseudo-gold), không fit tham số/thống kê nào của model. Kiểm chứng trên gold VSEC là tool-validation — không liên quan training.

## Nội dung
0. Cấu hình + tự dò input nb0
1. Cell hàm dùng chung — có `SHARED_CELLS_VERSION` để đối chiếu với nb3
2. Assert các case biên (split/merge/insert/delete/hoa-thường/dấu câu)
3. Kiểm chứng aligner trên gold VSEC — 3 tầng
4. Align test 6k
5. Roundtrip invariant (VSEC + test) — assert 100%
6. Thống kê (câu sạch, loại edit, suspect)
7. QA 50 mẫu in ra soát tay
8. Xuất `test_aligned.jsonl` + `qa_samples.json` + `align_report.json`

## 0. Cấu hình — mọi tham số gom một chỗ (DESIGN.md §7)

| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `SEED` | `42` | seed cho chọn mẫu QA (tái lập được) |
| `QA_N` | `50` | số mẫu QA in ra soát tay (DESIGN.md §2: ~50) |
| `SUSPECT_EDIT_RATIO` | `0.3` | ngưỡng đánh dấu câu *suspect*: edit quá dày so với độ dài câu → pseudo-annotation khó tin, thống kê riêng |

**Input tự dò**: quét `/kaggle/input/**/manifest.json` (Kaggle) rồi `./out/manifest.json` (local) — lấy thư mục đầu tiên có đủ `manifest.json` + `vsec_train.jsonl` + `vsec_val.jsonl` + `test_normalized.jsonl`. **Thiếu file → dừng sớm** với hướng dẫn sửa (nb0 phải chạy khi CÓ bộ test 6k thì mới sinh ra `test_normalized.jsonl` — nb0 §6).

In [ ]:
import datetime
import json
from pathlib import Path

SEED = 42
QA_N = 50
SUSPECT_EDIT_RATIO = 0.3

REQUIRED_FILES = ['manifest.json', 'vsec_train.jsonl', 'vsec_val.jsonl', 'test_normalized.jsonl']


def find_nb0_output():
    """Tự dò thư mục output của nb0: /kaggle/input/**/manifest.json rồi ./out/manifest.json."""
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('manifest.json'))
    local = Path('./out/manifest.json')
    if local.is_file():
        candidates.append(local)
    for m in candidates:
        if all((m.parent / f).is_file() for f in REQUIRED_FILES):
            return m.parent
    return None


INPUT_DIR = find_nb0_output()
if INPUT_DIR is None:
    raise FileNotFoundError(
        'Không tìm thấy output của nb0 (cần đủ: ' + ', '.join(REQUIRED_FILES) + '). '
        'Trên Kaggle: Add Input dataset đã tạo từ nb0 (xem nb0 §9). Local: chạy nb0 với OUTPUT_DIR ./out. '
        'Lưu ý: nb0 phải chạy khi CÓ bộ test 6k thì output mới có test_normalized.jsonl (nb0 §6).'
    )

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')
manifest_nb0 = json.loads((INPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('nb0 manifest:', manifest_nb0.get('notebook'), '— tạo', manifest_nb0.get('created'))
print(f'SEED={SEED}  QA_N={QA_N}  SUSPECT_EDIT_RATIO={SUSPECT_EDIT_RATIO}')
print('Số câu theo manifest nb0 — VSEC sau dedupe:', manifest_nb0.get('vsec', {}).get('after_dedupe'),
      '· test sau dedupe:', manifest_nb0.get('test', {}).get('after_dedupe'))

## 1. Cell hàm dùng chung — copy NGUYÊN VẸT sang nb3 (DESIGN.md §7)

nb3 sẽ copy cell này y hệt (không sửa) để (a) align `source ↔ model output` khi eval và (b) tự tính lại pseudo-annotation cho VSEC-val (quyết định 22/09: nb1 **không xuất** `vsec_*_aligned.jsonl`). Hằng số `SHARED_CELLS_VERSION` in ra khi chạy — nếu nb3 in khác version nghĩa là 2 notebook đã lệch nhau; lúc đó phải đồng bộ lại hoặc trích `.py` chung (nguyên tắc DESIGN.md §7: sửa ở ≥2 nơi thì trích module).

Hàm chính:

- `nfc_normalize(s)` — copy y hệt từ nb0 (NFC + strip + gộp khoảng trắng).
- `canon_tokenize(s)` — **token canonical**: NFC rồi tách bằng `re.findall(r'\w+|[^\w\s]+')` — run chữ/số liên liền là 1 token, cụm dấu câu liền kề là 1 token riêng. Đây là cách chuẩn hóa "dấu câu tách space như VSEC" (DESIGN.md §2.4) áp dụng **thống nhất cho VSEC / test / output model**, đưa cả 3 về cùng một hệ tọa độ. So sánh token **case-sensitive** (`tronG` ≠ `trong` — lỗi hoa/thường là lỗi thật trong bộ test).
- `levenshtein_opcodes(src, tgt)` — DP thuần Python (sub/ins/del = 1) có backtrace, trả opcodes kiểu difflib. Không thêm dependency.
- `extract_edit_blocks(...)` — gộp các opcode không-`equal` **liền kề** thành 1 edit block → chịu được split/merge/dịch vị trí (chỗ dễ sai nhất theo DESIGN.md §6).
- `build_pseudo_annotation(text, corrected)` — dựng annotation schema giống VSEC.
- `apply_edit_blocks(src, blocks)` — dùng cho roundtrip invariant (§5).

Loại block (theo số token nguồn → đích); `position` luôn là **index trong hệ token canonical**:

| type | nguồn → đích | ví dụ |
|---|---|---|
| `substitute` | 1 → 1 | `sanh` → `xanh` |
| `split` | 1 → n | `vàahệ` → `và hệ` (từ dính, sửa tách ra) |
| `merge` | m → 1 | `nh iên` → `nhiên` (âm tiết bị tách sai, sửa dính lại) |
| `insert` | 0 → n | nguồn thiếu âm tiết: `đi học` → `đi rồi học` |
| `delete` | m → 0 | nguồn thừa âm tiết: `Anh rồi đi` → `Anh đi` |
| `multi` | m → n | cụm phức tạp / lỗi kề nhau gộp block |

Quy ước quan trọng (chi tiết trong docstring `build_pseudo_annotation`):

- **`error_positions`** chỉ chứa vị trí token nguồn của block **có** token nguồn. Block `insert` không có vị trí nguồn → chỉ nằm trong `correction_pairs` với `error = ''`, `position` = vị trí chèn trước (có thể == `len(src)`), và **không** làm token nguồn nào thành `is_correct = False`.
- **`punct_only`**: block mà cả 2 vế chỉ toàn dấu câu → giữ lại kèm flag, để lọc/báo cáo riêng ở nb3.
- **`suspect`**: `edit_ratio` (tổng token nằm trong edit, tính cả 2 vế, chia `max(len(src), len(tgt))`) vượt `SUSPECT_EDIT_RATIO` → chỉ đánh dấu, không loại.
- Eval ở nb3 so khớp theo **block-overlap** (block model chồng lấn block gold theo span), không so index tuyệt đối — chịu được split/merge/dịch vị trí.

In [ ]:
import re
import unicodedata

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')


def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()


def canon_tokenize(s):
    """NFC + tách token: run chữ/số liên liền (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng.
    Chuẩn hóa "style VSEC" (dấu câu tách space) triệt để — dùng THỐNG NHẤT cho VSEC / test / output
    model (DESIGN.md §2.4). So sánh token là case-sensitive ("tronG" ≠ "trong")."""
    return _TOKEN_RE.findall(nfc_normalize(s))


def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)


def levenshtein_opcodes(src, tgt):
    """Levenshtein DP thuần Python (sub/ins/del = 1) trên 2 chuỗi token.
    Trả về opcodes kiểu difflib: list (tag, i1, i2, j1, j2) với tag ∈ {equal, replace, insert, delete},
    phủ kín src[i1:i2] ↔ tgt[j1:j2], không chồng lấn, theo thứ tự xuất hiện."""
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)  # match / substitute
            if prev[j] + 1 < best:  # delete token nguồn
                best = prev[j] + 1
            if row[j - 1] + 1 < best:  # insert token đích
                best = row[j - 1] + 1
            row[j] = best
    ops = []

    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))

    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops


def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }


def extract_edit_blocks(src, tgt, opcodes):
    """Gộp các opcode không-equal liền kề thành 1 edit block (chịu được split/merge/dịch vị trí).
    type theo số token (nguồn → đích): substitute 1-1 · split 1→n · merge m→1 · insert 0→n ·
    delete m→0 · multi m→n. position = index token nguồn đầu tiên của block
    (insert: vị trí chèn trước, có thể == len(src))."""
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks


def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (nb3 phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }


def apply_edit_blocks(src, blocks):
    """Roundtrip invariant: áp edit blocks vào src tokens → phải thu về đúng tgt tokens."""
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out


print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

## 2. Kiểm thử case biên — assert

Các case từ PROJECT.md §3.2 + đặc điểm bộ test (nhìn từ `data/6000.csv`): âm tiết tách sai, từ dính, ký tự thừa đầu từ, câu sạch, lỗi hoa/thường (`tronG`), thiếu/thừa âm tiết, khác chỉ dấu câu. Mỗi case đồng thời assert **roundtrip** (`src + edit_blocks == tgt`).

In [ ]:
def _ann(text, corrected):
    a = build_pseudo_annotation(text, corrected)
    assert apply_edit_blocks(a['src_tokens'], a['edit_blocks']) == a['tgt_tokens'], (text, corrected)
    return a


# 1. Âm tiết tách sai: 2 token nguồn → 1 token đích (merge)
a = _ann('Tuy nh iên , anh ấy đến', 'Tuy nhiên , anh ấy đến')
assert [b['type'] for b in a['edit_blocks']] == ['merge'], a['edit_blocks']
assert a['error_positions'] == [1, 2]  # cả 2 token nguồn ("nh", "iên") đều thuộc vùng lỗi
assert a['correction_pairs'] == [{'error': 'nh iên', 'correction': 'nhiên', 'position': 1}]

# 2. Từ dính: 1 token nguồn → 2 token đích (split)
a = _ann('anh vàahệ em', 'anh và hệ em')
assert [b['type'] for b in a['edit_blocks']] == ['split'], a['edit_blocks']
assert a['error_positions'] == [1]

# 3. Ký tự thừa ở đầu từ — ở mức token là substitute 1-1
a = _ann('aNăng suất tăng', 'Năng suất tăng')
assert [b['type'] for b in a['edit_blocks']] == ['substitute']
assert a['correction_pairs'][0] == {'error': 'aNăng', 'correction': 'Năng', 'position': 0}

# 4. Substitution + token dấu câu độc lập (style VSEC: "sanh , sạch , đẹp")
a = _ann('giữ môi trường sanh , sạch , đẹp .', 'giữ môi trường xanh , sạch , đẹp .')
assert [b['type'] for b in a['edit_blocks']] == ['substitute']
assert a['edit_blocks'][0]['punct_only'] is False
assert a['error_positions'] == [3]

# 5. Câu sạch
a = _ann('câu này sạch , đúng chính tả .', 'câu này sạch , đúng chính tả .')
assert a['is_clean'] and a['error_positions'] == [] and a['correction_pairs'] == []
assert all(s['is_correct'] for s in a['syllable_annotations'])

# 6. Lỗi hoa/thường — case-sensitive nên là edit thật
a = _ann('chuyên tronG lĩnh vực', 'chuyên trong lĩnh vực')
assert [b['type'] for b in a['edit_blocks']] == ['substitute']

# 7. Thiếu âm tiết ở nguồn (insert): KHÔNG có error_positions, chỉ ở correction_pairs,
#    và không token nguồn nào bị đánh dấu is_correct=False
a = _ann('Anh đi học .', 'Anh đã đi học .')
assert [b['type'] for b in a['edit_blocks']] == ['insert']
assert a['error_positions'] == []
assert a['correction_pairs'] == [{'error': '', 'correction': 'đã', 'position': 1}]
assert all(s['is_correct'] for s in a['syllable_annotations'])

# 8. Thừa âm tiết ở nguồn (delete)
a = _ann('Anh rồi đi học .', 'Anh đi học .')
assert [b['type'] for b in a['edit_blocks']] == ['delete']
assert a['error_positions'] == [1]
assert a['correction_pairs'] == [{'error': 'rồi', 'correction': '', 'position': 1}]
assert a['syllable_annotations'][1]['is_correct'] is False
assert a['syllable_annotations'][1]['corrections'] == ['']

# 9. Khác chỉ dấu câu → block punct_only
a = _ann('Anh đến rồi .', 'Anh đến rồi ,')
assert [b['type'] for b in a['edit_blocks']] == ['substitute']
assert a['edit_blocks'][0]['punct_only'] is True

# 10. Tokenization canonical: dấu câu dính được tách, 2 vế về cùng hệ tọa độ
assert canon_tokenize('Anh đến,rồi.') == ['Anh', 'đến', ',', 'rồi', '.']
assert canon_tokenize('Đám "ong ve" của') == ['Đám', '"', 'ong', 've', '"', 'của']
assert canon_tokenize('Suzuki GSX-R 1000 2012') == ['Suzuki', 'GSX', '-', 'R', '1000', '2012']

print('Tất cả assert case biên PASS —', SHARED_CELLS_VERSION)

## 3. Kiểm chứng aligner trên gold VSEC — 3 tầng

Chạy align `text ↔ corrected_text` trên **toàn bộ VSEC** (train+val, từ output nb0) rồi so với gold:

- **(a) Count agreement**: số block == `error_count` gold, mọi câu. Lệch là bình thường khi 2 lỗi kề nhau gộp thành 1 block.
- **(b) Pair containment (2 chiều)**: gold pair khớp nếu `canon_tokenize(error)` là token-subsequence **liền kề** trong `src_tokens` của block nào đó và `canon_tokenize(correction)` nằm trong `tgt_tokens` của cùng block. Báo recall (gold→block) + precision (block→gold) + tỉ lệ câu perfect.
- **(c) Exact position**: `position` gold nằm trong `src_span` của block khớp — **chỉ tính trên câu có `len(ws_tokens) == len(canon_tokens)`** (tránh offset hệ thống do token canonical tách dấu câu dính như `đẹp.`).

Đây là **tool-validation** (không fit tham số nào) → không liên quan leakage. Kỳ vọng không 100%: phần lệch gồm cả lỗi gold (edge case annotation, PROJECT.md §3.2) lẫn giới hạn aligner — in mẫu lệch ra để soi tay phân biệt.

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def contains_contiguous(seq, sub):
    if not sub:
        return True
    L = len(sub)
    return any(seq[k:k + L] == sub for k in range(len(seq) - L + 1))


vsec_all = load_jsonl(INPUT_DIR / 'vsec_train.jsonl') + load_jsonl(INPUT_DIR / 'vsec_val.jsonl')
manifest_vsec_n = manifest_nb0.get('vsec', {}).get('after_dedupe')
if manifest_vsec_n is not None and len(vsec_all) != manifest_vsec_n:
    print(f'[CẢNH BÁO] Số câu VSEC ({len(vsec_all)}) ≠ manifest nb0 ({manifest_vsec_n}) — nb1 chỉ báo, không tự sửa nb0.')
else:
    print(f'VSEC: {len(vsec_all)} câu (train+val) — khớp manifest nb0')

tier = {
    'n_sentences': 0,
    'count_match': 0,                                          # tầng (a)
    'gold_pairs_total': 0, 'gold_pairs_matched': 0,            # tầng (b) — recall
    'blocks_total': 0, 'blocks_matched': 0,                     # tầng (b) — precision
    'pos_eligible_pairs': 0, 'pos_in_span': 0,                  # tầng (c)
    'perfect_sentences': 0,
}
mismatch_samples = []
vsec_pseudo = []  # list (record, annotation)

for rec in vsec_all:
    ann = build_pseudo_annotation(rec['text'], rec['corrected_text'])
    vsec_pseudo.append((rec, ann))
    blocks = ann['edit_blocks']
    gold_pairs = rec.get('correction_pairs') or []
    ws_len = len(nfc_normalize(rec['text']).split())
    canon_len = len(ann['src_tokens'])

    tier['n_sentences'] += 1
    if ann['error_count'] == rec.get('error_count'):
        tier['count_match'] += 1

    gold_matched = []
    block_matched = [False] * len(blocks)
    for gp in gold_pairs:
        e_toks = canon_tokenize(gp.get('error', ''))
        c_toks = canon_tokenize(gp.get('correction', ''))
        if not e_toks and not c_toks:
            continue  # gold pair rỗng lạ — bỏ qua, không vào mẫu số
        tier['gold_pairs_total'] += 1
        hit = None
        for bi, b in enumerate(blocks):
            if contains_contiguous(b['src_tokens'], e_toks) and contains_contiguous(b['tgt_tokens'], c_toks):
                hit = (bi, b)
                break
        if hit is None:
            gold_matched.append(False)
            continue
        bi, b = hit
        gold_matched.append(True)
        block_matched[bi] = True
        tier['gold_pairs_matched'] += 1
        if ws_len == canon_len and isinstance(gp.get('position'), int):
            tier['pos_eligible_pairs'] += 1
            if b['src_span'][0] <= gp['position'] < b['src_span'][1]:
                tier['pos_in_span'] += 1

    tier['blocks_total'] += len(blocks)
    tier['blocks_matched'] += sum(block_matched)
    if all(gold_matched) and all(block_matched) and ann['error_count'] == rec.get('error_count'):
        tier['perfect_sentences'] += 1
    elif len(mismatch_samples) < 20:
        mismatch_samples.append({
            'row_id': rec.get('row_id'), 'split': rec.get('split'), 'text': rec['text'][:100],
            'gold_error_count': rec.get('error_count'),
            'gold_pairs': [
                {'error': gp.get('error'), 'correction': gp.get('correction'), 'position': gp.get('position')}
                for gp in gold_pairs
            ],
            'pseudo_blocks': [
                {'type': b['type'], 'position': b['position'],
                 'src': ' '.join(b['src_tokens']) or '∅', 'tgt': ' '.join(b['tgt_tokens']) or '∅'}
                for b in blocks
            ],
        })

print('== Kiểm chứng aligner trên gold VSEC (3 tầng) ==')
print(f"(a) count agreement   : {tier['count_match']}/{tier['n_sentences']} câu ({tier['count_match'] / tier['n_sentences']:.1%})")
print(f"(b) recall gold pairs : {tier['gold_pairs_matched']}/{tier['gold_pairs_total']} ({tier['gold_pairs_matched'] / max(tier['gold_pairs_total'], 1):.1%})")
print(f"(b) precision blocks  : {tier['blocks_matched']}/{tier['blocks_total']} ({tier['blocks_matched'] / max(tier['blocks_total'], 1):.1%})")
print(f"(b) perfect sentences : {tier['perfect_sentences']}/{tier['n_sentences']} ({tier['perfect_sentences'] / tier['n_sentences']:.1%})")
print(f"(c) position in span  : {tier['pos_in_span']}/{tier['pos_eligible_pairs']} ({tier['pos_in_span'] / max(tier['pos_eligible_pairs'], 1):.1%}) — chỉ câu có len(ws)==len(canon)")
print()
print(f'Mẫu lệch (giữ tối đa {len(mismatch_samples)}) — soi để phân biệt lỗi gold vs lỗi aligner:')
for s in mismatch_samples:
    print(json.dumps(s, ensure_ascii=False))

## 4. Align test 6k

Load `test_normalized.jsonl`, đối chiếu số câu với `manifest.json` của nb0 (lệch → **chỉ báo cáo, không tự sửa nb0**), rồi align từng cặp `text ↔ corrected_text` → pseudo-annotation. `align_failed` tuyệt đối (một vế token hóa rỗng) được đếm riêng.

In [ ]:
test_records = load_jsonl(INPUT_DIR / 'test_normalized.jsonl')
manifest_test_n = manifest_nb0.get('test', {}).get('after_dedupe')
if manifest_test_n is not None and len(test_records) != manifest_test_n:
    print(f'[CẢNH BÁO] Số câu test ({len(test_records)}) ≠ manifest nb0 ({manifest_test_n}) — nb1 chỉ báo, không tự sửa nb0.')
else:
    print(f'Test: {len(test_records)} câu — khớp manifest nb0')

test_aligned = []
for rec in test_records:
    ann = build_pseudo_annotation(rec['text'], rec['corrected_text'])
    test_aligned.append({**rec, 'pseudo': ann})

n_fail = sum(1 for r in test_aligned if r['pseudo']['align_failed'])
print(f'Đã align {len(test_aligned)} câu · align_failed tuyệt đối: {n_fail}')

## 5. Roundtrip invariant — assert 100%

Với **mọi câu** (VSEC + test): thay các edit block vào token nguồn phải thu về đúng token đích, và mọi `error_positions` nằm trong range. Đây là kiểm chứng cấu trúc của aligner (phủ kín, không chồng lấn, index hợp lệ) trước khi pseudo-annotation được dùng làm gold cho eval ở nb3.

In [ ]:
n_checked = 0
for rec, ann in vsec_pseudo:
    assert apply_edit_blocks(ann['src_tokens'], ann['edit_blocks']) == ann['tgt_tokens'], rec.get('row_id')
    assert all(0 <= p < len(ann['src_tokens']) for p in ann['error_positions']), rec.get('row_id')
    n_checked += 1
for r in test_aligned:
    a = r['pseudo']
    assert apply_edit_blocks(a['src_tokens'], a['edit_blocks']) == a['tgt_tokens'], r.get('text', '')[:60]
    assert all(0 <= p < len(a['src_tokens']) for p in a['error_positions'])
    n_checked += 1
print(f'Roundtrip invariant PASS 100% — {n_checked} câu (VSEC + test): src + edit_blocks == tgt, mọi vị trí trong range.')

## 6. Thống kê bộ test sau align

Số liệu then chốt: **tỉ lệ câu sạch** (câu không phát sinh edit — trả lời câu hỏi treo DESIGN.md §11.1, và quyết định có thêm chỉ số "tỷ lệ giữ nguyên câu sạch" ở DESIGN.md §6 hay không), tỉ lệ suspect, phân bố loại edit, phân bố số lỗi/câu so với gold VSEC.

In [ ]:
import collections


def error_bins(n):
    return '0' if n == 0 else ('1' if n == 1 else ('2' if n == 2 else ('3' if n == 3 else '>=4')))


def summarize(records, ann_get):
    s = {
        'n': len(records),
        'clean': 0, 'clean_strict': 0, 'suspect': 0, 'align_failed': 0,
        'block_types': collections.Counter(), 'punct_only_blocks': 0, 'only_punct_sentences': 0,
        'error_bins': collections.Counter(),
    }
    for r in records:
        a = ann_get(r)
        s['clean'] += a['is_clean']
        s['clean_strict'] += a['is_clean'] and not a['suspect']
        s['suspect'] += a['suspect']
        s['align_failed'] += a['align_failed']
        s['block_types'].update(b['type'] for b in a['edit_blocks'])
        s['punct_only_blocks'] += sum(1 for b in a['edit_blocks'] if b['punct_only'])
        if a['edit_blocks'] and all(b['punct_only'] for b in a['edit_blocks']):
            s['only_punct_sentences'] += 1
        s['error_bins'][error_bins(a['error_count'])] += 1
    return s


test_sum = summarize(test_aligned, lambda r: r['pseudo'])
vsec_sum = summarize(vsec_pseudo, lambda t: t[1])
vsec_gold_bins = collections.Counter(error_bins(rec.get('error_count') or 0) for rec, _ in vsec_pseudo)

n = test_sum['n']
print('== Thống kê bộ test sau align ==')
print(f'Tổng số câu                  : {n}')
print(f"Câu sạch (0 edit)            : {test_sum['clean']} ({test_sum['clean'] / n:.1%})  ← trả lời DESIGN.md §11.1")
print(f"Câu sạch loại suspect        : {test_sum['clean_strict']} ({test_sum['clean_strict'] / n:.1%})")
print(f"Câu suspect (edit dày)       : {test_sum['suspect']} ({test_sum['suspect'] / n:.1%})")
print(f"Align failed tuyệt đối       : {test_sum['align_failed']}")
print(f"Câu chỉ khác dấu câu         : {test_sum['only_punct_sentences']} · block punct_only: {test_sum['punct_only_blocks']}")
print()
print('Phân bố loại edit block (test):')
for t, c in test_sum['block_types'].most_common():
    print(f'  {t:12s}: {c}')
print()
print('Phân bố số lỗi/câu — test (pseudo, số block) vs VSEC (gold):')
for b in ['0', '1', '2', '3', '>=4']:
    print(f"  {b:4s}: test {test_sum['error_bins'][b]:5d} ({test_sum['error_bins'][b] / n:.1%})"
          f" · VSEC {vsec_gold_bins[b]:5d} ({vsec_gold_bins[b] / vsec_sum['n']:.1%})")

## 7. QA 50 mẫu — in ra soát tay

Chọn mẫu **seeded** (`random.Random(SEED)`) stratified: clean / substitute / insert / delete / split / merge / punct_only / suspect — stratum thiếu thì thay bằng random. Khi soát tay từng mẫu cần kiểm:

1. Span `[...]` đánh dấu có đúng chỗ lỗi thật không?
2. `correction` của block có đúng với ý nghĩa câu đã sửa không?
3. Block `merge`/`split`/`multi` có phải từ dính/tách sai/cụm lỗi cần xử lý riêng không?
4. Câu `suspect` có phải align sai (2 vế không thật sự song song) hay bản thân cặp dữ liệu gốc đã lỗi?

Danh sách mẫu cũng được ghi vào `qa_samples.json` để đối chiếu lại sau.

In [ ]:
import random

STRATA_QUOTAS = [
    ('clean', 10), ('substitute', 10), ('insert', 5), ('delete', 5),
    ('split', 5), ('merge', 5), ('punct_only', 5), ('suspect', 5),
]


def stratum_tags(a):
    tags = set()
    if a['is_clean']:
        tags.add('clean')
    if a['suspect']:
        tags.add('suspect')
    for b in a['edit_blocks']:
        tags.add('punct_only' if b['punct_only'] else b['type'])
    return tags


rng = random.Random(SEED)
picked = {}
for stratum, quota in STRATA_QUOTAS:
    cands = [i for i, r in enumerate(test_aligned) if i not in picked and stratum in stratum_tags(r['pseudo'])]
    for i in rng.sample(cands, min(quota, len(cands))):
        picked[i] = stratum
if len(picked) < QA_N:
    rest = [i for i in range(len(test_aligned)) if i not in picked]
    for i in rng.sample(rest, min(QA_N - len(picked), len(rest))):
        picked[i] = 'random'


def render_marked(tokens, blocks, side):
    marked = set()
    for b in blocks:
        lo, hi = b['src_span'] if side == 'src' else b['tgt_span']
        marked.update(range(lo, hi))
    return ' '.join(f'[{t}]' if i in marked else t for i, t in enumerate(tokens))


qa_samples = []
for i in sorted(picked):
    r = test_aligned[i]
    a = r['pseudo']
    print(f"—— QA idx={i} · stratum={picked[i]} · clean={a['is_clean']} · suspect={a['suspect']} · edit_ratio={a['edit_ratio']}")
    print('  SRC:', render_marked(a['src_tokens'], a['edit_blocks'], 'src'))
    print('  TGT:', render_marked(a['tgt_tokens'], a['edit_blocks'], 'tgt'))
    for b in a['edit_blocks']:
        note = ' (punct_only)' if b['punct_only'] else ''
        src_s = ' '.join(b['src_tokens']) or '∅'
        tgt_s = ' '.join(b['tgt_tokens']) or '∅'
        print(f"    - {b['type']}@{b['position']}: {src_s} → {tgt_s}{note}")
    print()
    qa_samples.append({'test_index': i, 'stratum': picked[i], 'text': r['text'],
                       'corrected_text': r['corrected_text'], 'pseudo': a})

print(f'Đã in {len(qa_samples)} mẫu QA (mục tiêu {QA_N}, seed {SEED}) để soát tay.')

## 8. Xuất file + báo cáo

- `test_aligned.jsonl` — mọi trường của `test_normalized.jsonl` + pseudo-annotation theo **schema giống VSEC** (`error_positions`, `correction_pairs`, `syllable_annotations`, `error_count`, `has_errors` = không clean) + `edit_blocks`, tokens, flags (`is_clean`, `suspect`, `align_failed`, `edit_ratio`) + `shared_cells_version`.
- `qa_samples.json` — các mẫu QA kèm annotation đầy đủ.
- `align_report.json` — cấu hình, kết quả kiểm chứng 3 tầng trên gold VSEC, thống kê test, tham chiếu manifest nb0.

**Không xuất** `vsec_*_aligned.jsonl` (quyết định 22/09): nb3 tự tính lại pseudo-annotation cho VSEC-val bằng cell hàm dùng chung — đối chiếu `SHARED_CELLS_VERSION` cho khớp.

In [ ]:
def write_jsonl(path, records):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


export_records = []
for r in test_aligned:
    a = r['pseudo']
    export_records.append({
        **{k: v for k, v in r.items() if k != 'pseudo'},
        'has_errors': not a['is_clean'],
        'error_count': a['error_count'],
        'error_positions': a['error_positions'],
        'correction_pairs': a['correction_pairs'],
        'syllable_annotations': a['syllable_annotations'],
        'edit_blocks': a['edit_blocks'],
        'is_clean': a['is_clean'],
        'suspect': a['suspect'],
        'align_failed': a['align_failed'],
        'edit_ratio': a['edit_ratio'],
        'src_tokens': a['src_tokens'],
        'tgt_tokens': a['tgt_tokens'],
        'shared_cells_version': SHARED_CELLS_VERSION,
    })

write_jsonl(OUTPUT_DIR / 'test_aligned.jsonl', export_records)

with open(OUTPUT_DIR / 'qa_samples.json', 'w', encoding='utf-8') as f:
    json.dump({'seed': SEED, 'shared_cells_version': SHARED_CELLS_VERSION, 'samples': qa_samples},
              f, ensure_ascii=False, indent=2)

report = {
    'created': RUN_STAMP,
    'notebook': 'nb1_align_annotate',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {'seed': SEED, 'qa_n': QA_N, 'suspect_edit_ratio': SUSPECT_EDIT_RATIO},
    'nb0_manifest_ref': {
        'notebook': manifest_nb0.get('notebook'),
        'created': manifest_nb0.get('created'),
        'vsec_after_dedupe': manifest_vsec_n,
        'test_after_dedupe': manifest_test_n,
    },
    'roundtrip_invariant_ok': True,
    'vsec_gold_validation': {
        'n_sentences': tier['n_sentences'],
        'count_match': tier['count_match'],
        'gold_pairs_total': tier['gold_pairs_total'],
        'gold_pairs_matched': tier['gold_pairs_matched'],
        'blocks_total': tier['blocks_total'],
        'blocks_matched': tier['blocks_matched'],
        'perfect_sentences': tier['perfect_sentences'],
        'pos_eligible_pairs': tier['pos_eligible_pairs'],
        'pos_in_span': tier['pos_in_span'],
        'mismatch_samples_kept': len(mismatch_samples),
    },
    'test': {
        'n': test_sum['n'],
        'clean': test_sum['clean'],
        'clean_strict': test_sum['clean_strict'],
        'suspect': test_sum['suspect'],
        'align_failed': test_sum['align_failed'],
        'only_punct_sentences': test_sum['only_punct_sentences'],
        'punct_only_blocks': test_sum['punct_only_blocks'],
        'block_types': dict(test_sum['block_types']),
        'error_bins': dict(test_sum['error_bins']),
        'vsec_gold_error_bins': dict(vsec_gold_bins),
    },
    'qa': {'n_printed': len(qa_samples), 'strata': dict(STRATA_QUOTAS)},
    'files': ['test_aligned.jsonl', 'qa_samples.json', 'align_report.json'],
}
with open(OUTPUT_DIR / 'align_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('Đã ghi vào', OUTPUT_DIR)
print('Files:', report['files'])
print()
print(json.dumps(report, ensure_ascii=False, indent=2))

## 9. Đưa output sang nb2/nb3

1. Sau khi chạy xong trên Kaggle (*Save Version → Run All*): panel **Output** → chọn `test_aligned.jsonl`, `qa_samples.json`, `align_report.json` → **New Dataset** (ví dụ `vsec-align1`).
2. nb2/nb3: **Add Input** → chọn dataset đó (cùng với dataset output nb0).
3. nb3 khi copy cell hàm dùng chung phải giữ `SHARED_CELLS_VERSION` nguyên vẹn — nếu buộc phải sửa aligner thì tăng version ở cả 2 nơi và ghi vào nhật ký quyết định của DESIGN.md (nguyên tắc DESIGN.md §7: bị sửa ở ≥2 nơi thì trích `.py` chung).

Bước tiếp theo (DESIGN.md §4): nb2 — bảng âm tiết + non-word rate (Pilot 1), mô hình nhiễu từ `correction_pairs` train + nguồn câu sạch (Pilot 2).